In [1]:
!pip install transformers datasets accelerate scikit-learn nltk sentencepiece protobuf tiktoken


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\kathy\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import numpy as np
import torch
import nltk
import re

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import StratifiedKFold
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)
from datasets import Dataset

nltk.download('stopwords')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

C:\Users\kathy\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\kathy\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
df = pd.read_csv('ExioNAICS.csv')
print(f"Full dataset shape: {df.shape}")
print(f"\nColumns:\n{df.columns.tolist()}")

Full dataset shape: (20535, 29)

Columns:
['Company Name', 'Company Description', 'NAICS Code', 'NAICS Title', '2022 NAICS Code', '2022 NAICS Title', '2017 NAICS Code', '2017 NAICS Title', 'Description', 'NAICS_5 Code', 'NAICS_5 Title', 'NAICS_5 Description', 'NAICS_4 Code', 'NAICS_4 Title', 'NAICS_4 Description', 'NAICS_3 Code', 'NAICS_3 Title', 'NAICS_3 Description', 'NAICS_2 Code', 'NAICS_2 Title', 'NAICS_2 Description', 'ExioML', 'region', 'Value Added [M.EUR]', 'Employment [1000 p.]', 'GHG emissions [kg CO2 eq.]', 'Energy Carrier Net Total [TJ]', 'Year', 'Carbon Intensity']


In [4]:
df = df[['Company Name', 'Company Description', 'NAICS Code', 'NAICS Title']].copy()
df = df.dropna(subset=['Company Description', 'NAICS Code']).reset_index(drop=True)

df['NAICS Code'] = df['NAICS Code'].astype(int).astype(str)

print(f"Working dataset shape: {df.shape}")
print(f"Unique 6-digit NAICS codes: {df['NAICS Code'].nunique()}")
print(f"\nDescription length stats:")
print(df['Company Description'].str.len().describe())
print(f"\nSample rows:")
df.head(10)

Working dataset shape: (20535, 4)
Unique 6-digit NAICS codes: 1115

Description length stats:
count    20535.000000
mean       272.691794
std        113.070907
min         49.000000
25%        214.000000
50%        258.000000
75%        296.000000
max       1416.000000
Name: Company Description, dtype: float64

Sample rows:


,Company Name,Company Description,NAICS Code,NAICS Title
0,Foot Locker Specialty Inc,Foot Locker Inc (Foot Locker) is a specialty r...,448150,Clothing Accessories Stores
1,Cengage Learning Inc,Cengage is the education and technology compan...,511130,Book Publishers
2,Cardinal Scale Mfg Co,Cardinal Scale Manufacturing Company manufactu...,333997,Scale and Balance Manufacturing
3,Alticor Inc,Alticor Inc. owns and manages manufacturing an...,454390,Other Direct Selling Establishments
4,S Butler-Rosboro Corporation,Rosboro is North America's largest producer of...,335110,Electric Lamp Bulb and Part Manufacturing
5,Rs Legacy Corporation,RS Legacy Corporation was founded in 1963. The...,443142,Electronics Stores
6,Amway International Inc,"Amway International, Inc. wholesales and distr...",454390,Other Direct Selling Establishments
7,Carl Zeiss Inc,"Carl Zeiss, Inc. develops and manufactures opt...",333314,Optical Instrument and Lens Manufacturing
8,Iheartmedia Capital II LLC,"Lighthouse Document Technologies, Inc. provide...",517919,All Other Telecommunications
9,Jones Financial Companies Lllp,"The Jones Financial Companies, L.L.L.P. operat...",523120,Securities Brokerage


In [5]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_description'] = df['Company Description'].apply(preprocess_text)

print("Before vs After preprocessing:")
for i in range(3):
    print(f"\n--- Example {i+1} ---")
    print(f"Original:  {df['Company Description'].iloc[i][:120]}...")
    print(f"Cleaned:   {df['clean_description'].iloc[i][:120]}...")

Before vs After preprocessing:

--- Example 1 ---
Original:  Foot Locker Inc (Foot Locker) is a specialty retailer of athletic footwear and apparel....
Cleaned:   foot locker inc foot locker is a specialty retailer of athletic footwear and apparel...

--- Example 2 ---
Original:  Cengage is the education and technology company built for learners. Confident students are successful learners, so we de...
Cleaned:   cengage is the education and technology company built for learners confident students are successful learners so we desi...

--- Example 3 ---
Original:  Cardinal Scale Manufacturing Company manufactures and markets weighing products and systems. The Company offers baker do...
Cleaned:   cardinal scale manufacturing company manufactures and markets weighing products and systems the company offers baker dou...


In [6]:
label_counts = df['NAICS Code'].value_counts()

print(f"Total samples: {len(df)}")
print(f"Unique NAICS-6 codes: {df['NAICS Code'].nunique()}")
print(f"\nLabel frequency distribution:")
print(f"  Most common:  {label_counts.iloc[0]} samples (code: {label_counts.index[0]})")
print(f"  Least common: {label_counts.iloc[-1]} samples (code: {label_counts.index[-1]})")
print(f"  Median count:  {label_counts.median():.0f}")
print(f"  Mean count:    {label_counts.mean():.1f}")

print(f"\nTop 20 most frequent NAICS codes:")
print(label_counts.head(20))

print(f"\nCodes with only 1 sample: {(label_counts == 1).sum()}")
print(f"Codes with <= 5 samples:  {(label_counts <= 5).sum()}")

Total samples: 20535
Unique NAICS-6 codes: 1115

Label frequency distribution:
  Most common:  98 samples (code: 541330)
  Least common: 1 samples (code: 114119)
  Median count:  17
  Mean count:    18.4

Top 20 most frequent NAICS codes:
NAICS Code
541330    98
561110    79
533110    78
423830    73
336390    67
323111    65
424490    64
561990    55
541511    54
236220    52
551112    52
484121    52
325412    51
541512    50
325180    50
722511    50
311999    50
326199    50
541611    49
325998    48
Name: count, dtype: int64

Codes with only 1 sample: 2
Codes with <= 5 samples:  37


In [7]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")

token_lengths = df['clean_description'].apply(lambda x: len(tokenizer.encode(x)))

print("Token length distribution:")
print(token_lengths.describe())

for threshold in [64, 128, 256, 512]:
    pct = (token_lengths <= threshold).mean() * 100
    print(f"\n  max_length={threshold}: {pct:.1f}% of samples fit without truncation"
          f"  ({(token_lengths > threshold).sum()} would be truncated)")

Token length distribution:
count    20535.000000
mean        43.446262
std         18.515121
min          9.000000
25%         33.000000
50%         40.000000
75%         47.000000
max        274.000000
Name: clean_description, dtype: float64

  max_length=64: 92.6% of samples fit without truncation  (1516 would be truncated)

  max_length=128: 99.6% of samples fit without truncation  (87 would be truncated)

  max_length=256: 100.0% of samples fit without truncation  (2 would be truncated)

  max_length=512: 100.0% of samples fit without truncation  (0 would be truncated)


choose max_length = 128

In [12]:
le = LabelEncoder()
df['label'] = le.fit_transform(df['NAICS Code'])

num_labels = len(le.classes_)
print(f"Number of classes: {num_labels}")
print(f"Label range: {df['label'].min()} to {df['label'].max()}")

print(f"\nExample mappings (NAICS Code -> encoded label):")
for i in range(5):
    print(f"  '{df['NAICS Code'].iloc[i]}' ({df['NAICS Title'].iloc[i]}) -> {df['label'].iloc[i]}")

print(f"\nReverse mapping check:")
print(f"  Label 0 -> NAICS code '{le.inverse_transform([0])[0]}'")
print(f"  Label {num_labels-1} -> NAICS code '{le.inverse_transform([num_labels-1])[0]}'")

Number of classes: 1115
Label range: 0 to 1114

Example mappings (NAICS Code -> encoded label):
  '448150' (Clothing Accessories Stores) -> 630
  '511130' (Book Publishers) -> 744
  '333997' (Scale and Balance Manufacturing) -> 400
  '454390' (Other Direct Selling Establishments) -> 661
  '335110' (Electric Lamp Bulb and Part Manufacturing) -> 428

Reverse mapping check:
  Label 0 -> NAICS code '111110'
  Label 1114 -> NAICS code '928120'


In [13]:
df.head()

,Company Name,Company Description,NAICS Code,NAICS Title,clean_description,label
0,Foot Locker Specialty Inc,Foot Locker Inc (Foot Locker) is a specialty r...,448150,Clothing Accessories Stores,foot locker inc foot locker is a specialty ret...,630
1,Cengage Learning Inc,Cengage is the education and technology compan...,511130,Book Publishers,cengage is the education and technology compan...,744
2,Cardinal Scale Mfg Co,Cardinal Scale Manufacturing Company manufactu...,333997,Scale and Balance Manufacturing,cardinal scale manufacturing company manufactu...,400
3,Alticor Inc,Alticor Inc. owns and manages manufacturing an...,454390,Other Direct Selling Establishments,alticor inc owns and manages manufacturing and...,661
4,S Butler-Rosboro Corporation,Rosboro is North America's largest producer of...,335110,Electric Lamp Bulb and Part Manufacturing,rosboro is north america s largest producer of...,428


In [ ]:
export_df = df[['clean_description', 'NAICS Code', 'label']].copy()
export_df.to_csv('ExioNAICS_preprocessed.csv', index=False)

print(f"Exported preprocessed data to 'ExioNAICS_preprocessed.csv'")
print(f"Shape: {export_df.shape}")
print(f"Columns: {export_df.columns.tolist()}")
export_df.head()

## Hyperparameter Configuration

Change `SESSION` to 1, 2, or 3 before each Colab run. Each session tests a different training strategy.

In [ ]:
SESSION = 1  # 1,2,3

SESSION_CONFIGS = {
    1: {"learning_rate": 2e-5, "batch_size": 16, "weight_decay": 0.01, "name": "conservative"},
    2: {"learning_rate": 3e-5, "batch_size": 32, "weight_decay": 0.01, "name": "fast_convergence"},
    3: {"learning_rate": 1e-5, "batch_size": 16, "weight_decay": 0.1,  "name": "slow_strong_reg"},
}

config = SESSION_CONFIGS[SESSION]

MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LENGTH = 128
NUM_EPOCHS = 15
WARMUP_RATIO = 0.1
EARLY_STOPPING_PATIENCE = 3
N_FOLDS = 5
SEED = 42

print(f"=== Session {SESSION}: {config['name']} ===")
print(f"  Learning rate:  {config['learning_rate']}")
print(f"  Batch size:     {config['batch_size']}")
print(f"  Weight decay:   {config['weight_decay']}")
print(f"  Max length:     {MAX_LENGTH}")
print(f"  Epochs:         {NUM_EPOCHS} (with early stopping, patience={EARLY_STOPPING_PATIENCE})")
print(f"  Folds:          {N_FOLDS}")
print(f"  Warmup ratio:   {WARMUP_RATIO}")

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

folds = list(skf.split(df['clean_description'], df['label']))

for i, (train_idx, val_idx) in enumerate(folds):
    train_labels = df['label'].iloc[train_idx]
    val_labels = df['label'].iloc[val_idx]
    print(f"Fold {i+1}: train={len(train_idx)}, val={len(val_idx)}, "
          f"train classes={train_labels.nunique()}, val classes={val_labels.nunique()}")

In [ ]:
def tokenize_data(texts, labels, tokenizer, max_length):
    encodings = tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=max_length,
        return_tensors=None,
    )
    dataset = Dataset.from_dict({
        "input_ids": encodings["input_ids"],
        "attention_mask": encodings["attention_mask"],
        "labels": labels,
    })
    return dataset

print("Tokenization function defined.")
print(f"Using tokenizer: {MODEL_NAME}, max_length: {MAX_LENGTH}")

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    top1_preds = np.argmax(logits, axis=1)

    top1_acc = accuracy_score(labels, top1_preds)
    macro_f1 = f1_score(labels, top1_preds, average='macro', zero_division=0)
    weighted_f1 = f1_score(labels, top1_preds, average='weighted', zero_division=0)

    top5_acc = np.mean([
        1 if label in np.argsort(logit)[-5:] else 0
        for logit, label in zip(logits, labels)
    ])
    top10_acc = np.mean([
        1 if label in np.argsort(logit)[-10:] else 0
        for logit, label in zip(logits, labels)
    ])

    return {
        "top1_accuracy": top1_acc,
        "top5_accuracy": top5_acc,
        "top10_accuracy": top10_acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
    }

print("Metrics function defined: Top-1, Top-5, Top-10 accuracy, Macro F1, Weighted F1")

In [ ]:
import json, os, gc

all_fold_results = []

for fold_idx, (train_idx, val_idx) in enumerate(folds):
    print(f"\n{'='*60}")
    print(f"  FOLD {fold_idx+1}/{N_FOLDS} — Session {SESSION} ({config['name']})")
    print(f"{'='*60}")

    train_texts = df['clean_description'].iloc[train_idx].tolist()
    val_texts = df['clean_description'].iloc[val_idx].tolist()
    train_labels = df['label'].iloc[train_idx].tolist()
    val_labels = df['label'].iloc[val_idx].tolist()

    train_dataset = tokenize_data(train_texts, train_labels, tokenizer, MAX_LENGTH)
    val_dataset = tokenize_data(val_texts, val_labels, tokenizer, MAX_LENGTH)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=num_labels,
    )

    output_dir = f"./results/session_{SESSION}_fold_{fold_idx+1}"

    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=config['batch_size'],
        per_device_eval_batch_size=64,
        learning_rate=config['learning_rate'],
        weight_decay=config['weight_decay'],
        warmup_ratio=WARMUP_RATIO,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="top1_accuracy",
        greater_is_better=True,
        save_total_limit=1,
        logging_steps=50,
        fp16=True,
        seed=SEED,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    )

    trainer.train()

    eval_results = trainer.evaluate()
    eval_results["fold"] = fold_idx + 1
    all_fold_results.append(eval_results)

    print(f"\nFold {fold_idx+1} results:")
    print(f"  Top-1 Accuracy: {eval_results['eval_top1_accuracy']:.4f}")
    print(f"  Top-5 Accuracy: {eval_results['eval_top5_accuracy']:.4f}")
    print(f"  Top-10 Accuracy: {eval_results['eval_top10_accuracy']:.4f}")
    print(f"  Macro F1:       {eval_results['eval_macro_f1']:.4f}")
    print(f"  Weighted F1:    {eval_results['eval_weighted_f1']:.4f}")

    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()

print(f"\n{'='*60}")
print(f"  ALL FOLDS COMPLETE — Session {SESSION}")
print(f"{'='*60}")

In [ ]:
metrics_keys = ["eval_top1_accuracy", "eval_top5_accuracy", "eval_top10_accuracy",
                "eval_macro_f1", "eval_weighted_f1"]

print(f"\n=== Session {SESSION} ({config['name']}) — Summary ===\n")
print(f"{'Metric':<22} {'Mean':>8} {'Std':>8}  Per-fold values")
print("-" * 75)

summary = {"session": SESSION, "config": config}
for key in metrics_keys:
    values = [r[key] for r in all_fold_results]
    mean_val = np.mean(values)
    std_val = np.std(values)
    fold_str = ", ".join([f"{v:.4f}" for v in values])
    print(f"{key:<22} {mean_val:>8.4f} {std_val:>8.4f}  [{fold_str}]")
    summary[key] = {"mean": mean_val, "std": std_val, "per_fold": values}

os.makedirs("results", exist_ok=True)
results_file = f"results/session_{SESSION}_{config['name']}.json"
with open(results_file, "w") as f:
    json.dump(summary, f, indent=2, default=str)
print(f"\nResults saved to: {results_file}")